# 第 2 章：封装与属性控制

> 本章目标：理解**封装（Encapsulation）**的意义，掌握 Python 的访问控制约定（`public` / `_protected` / `__private`）、`@property` 属性装饰器，以及 `__slots__` 内存优化。

---

## 2.1 什么是封装？

**封装 = 隐藏内部实现细节，只暴露受控的接口。**

类比：你开汽车时只需要方向盘和油门，不需要理解发动机怎么点火。如果别人能直接把手伸进发动机乱拧，车迟早要坏。

```mermaid
flowchart LR
    subgraph 类的内部
        DATA[内部数据<br/>_balance] 
        LOGIC[校验逻辑<br/>余额不能为负]
    end
    USER[外部代码] -->|只能通过公开接口| API[deposit / withdraw]
    API --> LOGIC
    LOGIC --> DATA
    USER -.->|禁止直接乱改| DATA
```

封装带来的好处：
1. **保护数据**：防止外部把对象改成一个非法状态；
2. **降低耦合**：内部实现可以随时改，只要接口不变，调用方无感知；
3. **便于维护**：读写数据的地方集中，排查问题容易。

## 2.2 Python 的访问控制：靠"约定"而非"强制"

> 💡 **对比 Java/C++**：它们有真正的 `private` 关键字，编译器强制拦截访问。
> Python 没有真正的私有——它信奉 "we are all consenting adults"（我们都是理智的成年人），用**命名约定**表达意图。

| 写法 | 名称 | 含义 | 强制力 |
|------|------|------|--------|
| `name` | 公有 | 谁都能访问 | 无限制 |
| `_name` | 受保护 | "内部使用，别碰" 的**约定**，IDE 会提示 | 无强制，靠自觉 |
| `__name` | 私有 | 触发**名称改写（name mangling）**，变成 `_类名__name` | 半强制（仍可绕过） |

In [1]:
class BankAccount:
    bank_name = "宇宙银行"          # 公有

    def __init__(self, owner, balance=0):
        self.owner = owner         # 公有
        self._balance = balance    # 受保护：约定外部不要直接改
        self.__password = "123456" # 私有：触发名称改写

    def check_password(self, pwd):
        return pwd == self.__password   # 类内部可以正常访问


acc = BankAccount("张三", 1000)
print(acc.owner)              # ✅ 公有，随便访问
print(acc._balance)           # ⚠️ 能访问，但违反约定（IDE 会灰显/警告）

# print(acc.__password)       # ❌ AttributeError！
# 真相：__password 被改写成了 _BankAccount__password
print(acc._BankAccount__password)   # 仍能访问——Python 没有真正的私有

# 查看所有属性，注意名称改写的结果
print([k for k in acc.__dict__])

张三
1000
123456
['owner', '_balance', '_BankAccount__password']


## 2.3 名称改写（Name Mangling）的真正用途

`__attr` 的设计目的**不是**为了保密，而是为了**避免子类意外覆盖父类的内部属性**：

```mermaid
classDiagram
    class Parent {
        -__cache 实际叫 _Parent__cache
    }
    class Child {
        -__cache 实际叫 _Child__cache
    }
    Parent <|-- Child
```

因为改后的名字带有类名前缀，父子的 `__cache` 是**两个不同的属性**，互不干扰。

In [2]:
class Parent:
    def __init__(self):
        self.__cache = "父类的缓存"

    def show(self):
        print(f"Parent 视角: {self.__cache}")


class Child(Parent):
    def __init__(self):
        super().__init__()
        self.__cache = "子类的缓存"   # 不会覆盖父类的 __cache！


c = Child()
c.show()                      # 父类方法读到的还是父类的缓存
print(c.__dict__)             # 两个改写过后的名字并存

Parent 视角: 父类的缓存
{'_Parent__cache': '父类的缓存', '_Child__cache': '子类的缓存'}


## 2.4 `@property`：把方法伪装成属性

场景：你想在**读取/修改属性时做校验或计算**，但又不想让调用方写 `get_balance()` / `set_balance(x)` 这种啰嗦的方法调用。

> 💡 **对比 Java**：Java 必须先写 getter/setter 占位，因为以后加校验时不能改接口；
> Python 可以先暴露普通属性，未来需要校验时**无缝切换**为 `@property`，调用方代码一行都不用改。

```mermaid
flowchart LR
    A[外部代码: acc.balance = -100] --> B[@balance.setter]
    B --> C{校验: >= 0 ?}
    C -- 通过 --> D[写入 _balance]
    C -- 拒绝 --> E[抛出 ValueError]
    F[外部代码: print acc.balance] --> G[@property getter]
    G --> H[返回 _balance]
```

In [3]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.balance = balance    # 走的是 setter，自带校验

    @property
    def balance(self):
        """读取余额（像访问属性一样）"""
        return self._balance

    @balance.setter
    def balance(self, value):
        """设置余额（自动校验）"""
        if not isinstance(value, (int, float)):
            raise TypeError("余额必须是数字")
        if value < 0:
            raise ValueError("余额不能为负数")
        self._balance = value

    @balance.deleter
    def balance(self):
        """del acc.balance 时触发"""
        print("余额被清零")
        self._balance = 0


acc = BankAccount("张三", 1000)
print(acc.balance)      # 读：像属性
acc.balance = 500       # 写：像属性，其实调用了 setter
print(acc.balance)

try:
    acc.balance = -1    # 触发校验
except ValueError as e:
    print(f"拦截成功: {e}")

del acc.balance         # 触发 deleter
print(acc.balance)

1000
500
拦截成功: 余额不能为负数
余额被清零
0


## 2.5 只读属性与计算属性

`@property` 的两个高频用法：
1. **只读属性**：只写 getter 不写 setter，外部无法修改；
2. **计算属性**：值不存储，每次访问时动态计算（如由摄氏温度算出华氏温度）。

In [4]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value <= 0:
            raise ValueError("半径必须为正数")
        self._radius = value

    @property
    def area(self):
        """计算属性：不存储，实时计算"""
        import math
        return math.pi * self._radius ** 2


c = Circle(5)
print(f"半径={c.radius}, 面积={c.area:.2f}")
c.radius = 10                # 修改半径
print(f"半径={c.radius}, 面积={c.area:.2f}")  # 面积自动跟着变

try:
    c.area = 100             # ❌ 没有 setter → AttributeError，天然只读
except AttributeError as e:
    print(f"只读保护: {e}")

半径=5, 面积=78.54
半径=10, 面积=314.16
只读保护: property 'area' of 'Circle' object has no setter


## 2.6 `__slots__`：限制属性 + 节省内存

默认情况下，每个实例都有一个 `__dict__` 字典来存属性——灵活（可以随时加新属性）但占内存。

`__slots__` 声明"这个类只允许有这些属性"，实例**不再有 `__dict__`**：

| 对比 | 普通类 | 使用 `__slots__` |
|------|--------|------------------|
| 属性存储 | `__dict__` 字典 | 固定槽位（类似数组） |
| 动态加属性 | ✅ 可以 | ❌ 报 AttributeError |
| 内存占用 | 较高 | 显著降低（实例多时可省 40%+） |
| 适用场景 | 常规开发 | 需要创建**百万级**实例的类 |

In [5]:
import sys

class NormalPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class SlottedPoint:
    __slots__ = ("x", "y")   # 只允许这两个属性

    def __init__(self, x, y):
        self.x = x
        self.y = y


p1 = NormalPoint(1, 2)
p2 = SlottedPoint(1, 2)

p1.z = 3                     # ✅ 普通类可动态加属性
print(f"普通类有 __dict__: {p1.__dict__}")

try:
    p2.z = 3                 # ❌ slots 类不允许
except AttributeError as e:
    print(f"slots 限制: {e}")

print(f"普通类实例大小: {sys.getsizeof(p1) + sys.getsizeof(p1.__dict__)} 字节(含 __dict__)")
print(f"slots 实例大小: {sys.getsizeof(p2)} 字节")

# ⚠️ 注意：使用 __slots__ 的类，若父类没有 __slots__，效果会被抵消（父类的 __dict__ 还在）

普通类有 __dict__: {'x': 1, 'y': 2, 'z': 3}
slots 限制: 'SlottedPoint' object has no attribute 'z'
普通类实例大小: 344 字节(含 __dict__)
slots 实例大小: 48 字节


## 2.7 本章小结

| 机制 | 作用 | 一句话记忆 |
|------|------|-----------|
| `_attr` | 受保护约定 | "内部用的，别碰" |
| `__attr` | 名称改写 | 防子类意外覆盖，不是保密 |
| `@property` | 属性化方法 | 读写时夹带校验/计算逻辑 |
| 只写 getter | 只读属性 | 天然防止外部修改 |
| `__slots__` | 固定属性集 | 省内存，禁动态加属性 |

### 📝 动手练习

1. 给第 1 章的 `BankAccount` 加上 `@property`：`balance` 只读，只能通过 `deposit`/`withdraw` 改变。
2. 写一个 `Temperature` 类，内部存摄氏温度 `_celsius`，提供 `celsius` 和 `fahrenheit` 两个可读写的 property（华氏 = 摄氏 × 9/5 + 32），改任意一个另一个自动同步。
3. 思考题：`@property` 的方法每次访问都会执行，如果计算很耗时，有什么优化思路？（提示：`functools.cached_property`）

---
**下一章** 👉 `03_继承与方法解析顺序MRO.ipynb`：学习代码复用的核心机制——继承，以及多继承下的方法查找规则。